# 📦 W7: 재고 알림 자동화
**hexa-3 — W07** | ⏱️ 60분 | 🎯 재고 부족 상품 자동 감지 + Gemini 발주 메시지 생성 + 웹훅 fallback

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib numpy
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, pandas as pd, numpy as np
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
# === 2026년 3월 기준 최신 모델 선택 ===
# ✅ GA(안정): gemini-2.5-flash  ← 기본값 (권장)
# 🔵 Preview : gemini-3.1-flash-lite-preview  (최신, 2026.3 출시)
# 🔵 Preview : gemini-3.1-pro-preview  (최강, 복잡한 분석용)
# ⚠️  구버전 gemini-2.0-flash 는 2026년 6월 1일 종료 예정
MODEL_NAME = 'gemini-2.5-flash'  # 원하는 모델로 변경하세요
model=genai.GenerativeModel('gemini-2.5-flash'); print('✅ 준비 완료')


## Step 1: 쇼핑몰 정보 입력 (✏️)

In [ ]:
STORE={'name':'✏️ [쇼핑몰명]','platform':'✏️ [스마트스토어|쿠팡|자사몰]'}
print('✅',STORE['name'])


## Step 2: 샘플 데이터 준비

In [ ]:
np.random.seed(42)
dates=pd.date_range('2026-01-01',periods=30)
sales_data=pd.DataFrame({'날짜':dates.strftime('%Y-%m-%d'),
    '매출':np.random.randint(300000,1500000,30),'주문수':np.random.randint(30,150,30)})
sales_data.loc[7,'매출']=5000000; sales_data.loc[20,'매출']=8000000  # 이상치 보장
print(sales_data.describe())


## Step 3: 재고 부족 상품 자동 감지 + Gemini 발주 메시지 생성 + 웹훅 fallback

In [ ]:
p=f"""재고 부족 상품 자동 감지 + Gemini 발주 메시지 생성 + 웹훅 fallback: 소매업 AI 컨설턴트로서 수행하세요.
쇼핑몰: {{STORE['name']}} ({{STORE['platform']}})
인사이트 3가지 + 즉시 실행 액션 + 기대 ROI. 마크다운."""
resp=model.generate_content(p)
print(resp.text)


## Step 4: 시각화

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(range(len(sales_data)),sales_data['매출'],color='steelblue',linewidth=2)
ax.plot(range(len(sales_data)),sales_data['매출'].rolling(7,min_periods=1).mean(),
        color='red',linestyle='--',label='7일 평균')
ax.set_title(f'{STORE["name"]} — {title}')
ax.set_ylabel('매출(원)'); ax.legend()
plt.tight_layout(); plt.savefig('chart.png',dpi=150,bbox_inches='tight'); plt.show()


## Step 5: 다운로드

In [ ]:
with open('report.md','w',encoding='utf-8') as f: f.write(resp.text)
sales_data.to_csv('data.csv',index=False,encoding='utf-8-sig')
from google.colab import files
files.download('report.md'); files.download('data.csv'); files.download('chart.png')
print('✅ 완료!')


---
## 🔥 확장 과제
1. 실제 매출 데이터로 교체
2. 다음 주차와 연계